In [25]:
import sklearn
import numpy as np
import matplotlib as mlt
import pandas as pd

In [26]:
df = pd.read_csv('long_beach_california_cats_with_tnr_all_years.csv')

In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27418 entries, 0 to 27417
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   kennel_identity           27418 non-null  int64  
 1   animal_id                 27418 non-null  str    
 2   animal_name               16881 non-null  str    
 3   animal_type               27418 non-null  str    
 4   primary_color             27418 non-null  str    
 5   secondary_color           10996 non-null  str    
 6   sex                       27418 non-null  str    
 7   dob                       26347 non-null  str    
 8   intake_date               27418 non-null  str    
 9   intake_cond               27418 non-null  str    
 10  intake_type               27418 non-null  str    
 11  intake_subtype            27112 non-null  str    
 12  reason                    1328 non-null   str    
 13  outcome_date              27231 non-null  str    
 14  crossing         

In [28]:
text_cols = df.select_dtypes(exclude=['number', 'datetime', 'datetimetz'])

for col in text_cols.columns:
    unique_vals = text_cols[col].unique()
    print(f"Column: {col} ({len(unique_vals)} unique values)")
    print(f"Values: {unique_vals}\n" + "-"*40)


Column: animal_id (26722 unique values)
Values: <StringArray>
['A663398', 'A663399', 'A663400', 'A632381', 'A615510', 'A607321', 'A740168',
 'A670718', 'A704660', 'A717137',
 ...
 'A673493', 'A676826', 'A626046', 'A633252', 'A650783', 'A657773', 'A656354',
 'A643878', 'A666706', 'A632504']
Length: 26722, dtype: str
----------------------------------------
Column: animal_name (9440 unique values)
Values: <StringArray>
[         '*LOCK',         '*SHOCK',        '*BARREL',              nan,
         '*NICKY',        '*TOOTIE',        '*XANDER',       '*CALHOUN',
        '*POTATO',        '*CONCHA',
 ...
        '*CATIUS',       '*SCALLOP',          'LUNAR',          '*ALIE',
 'A633252*SHELIA',        '*DERICK',          'JULIE',   'MISS PUMPKIN',
         '*GUMBO',           'YUGI']
Length: 9440, dtype: str
----------------------------------------
Column: animal_type (1 unique values)
Values: <StringArray>
['CAT']
Length: 1, dtype: str
----------------------------------------
Column: pri

In [29]:
# The [78] means "must be a 7 or an 8", and \d{2} means "followed by 2 more digits"
real_zip_pattern = r'\b(90[78]\d{2})\b'
zip_codes = df['crossing'].str.extract(real_zip_pattern)

# Isolate the extracted series and drop missing matches
zip_series = zip_codes[0].dropna()

# Get the clean, complete value counts
zip_counts = zip_series.value_counts()

print(f"Real unique ZIP count: {len(zip_counts)}\n")
print("Counts per ZIP code:")
print(zip_counts.to_string())


Real unique ZIP count: 35

Counts per ZIP code:
0
90805    5797
90813    3579
90806    2554
90703    1854
90810    1761
90815    1640
90804    1434
90802    1263
90808    1252
90807    1172
90755     606
90803     456
90814     417
90720     344
90740     111
90840      25
90706      23
90715      19
90712      17
90801      16
90713      13
90716      10
90744       7
90701       6
90822       6
90743       6
90731       5
90723       4
90732       4
90745       2
90746       2
90705       2
90730       1
90846       1
90702       1


In [30]:
print(df.head())

   kennel_identity animal_id animal_name animal_type primary_color  \
0           328112   A663398       *LOCK         CAT          GRAY   
1           328113   A663399      *SHOCK         CAT          GRAY   
2           328114   A663400     *BARREL         CAT          GRAY   
3           307917   A632381         NaN         CAT     BRN TABBY   
4           299286   A615510         NaN         CAT        TORTIE   

  secondary_color       sex         dob intake_date       intake_cond  ...  \
0           WHITE  Neutered  2021-06-07  2021-09-07            NORMAL  ...   
1           WHITE  Neutered  2021-06-07  2021-09-07            NORMAL  ...   
2           WHITE  Neutered  2021-06-07  2021-09-07            NORMAL  ...   
3             NaN    Female  2019-08-08  2019-08-22  UNDER AGE/WEIGHT  ...   
4       BRN TABBY    Female  2018-09-03  2018-10-03  UNDER AGE/WEIGHT  ...   

    intake_is_dead                  geopoint intake_duration is_current_month  \
0  Alive on Intake  33.809718

In [31]:
print(df.tail())

       kennel_identity animal_id   animal_name animal_type primary_color  \
27413           322855   A656354  MISS PUMPKIN         CAT    CALICO TAB   
27414           315909   A643878        *GUMBO         CAT    GRAY TABBY   
27415           330311   A666706          YUGI         CAT     BLK TABBY   
27416           307993   A632504           NaN         CAT        TORTIE   
27417           292867   A593510       *DALLAS         CAT         BLACK   

      secondary_color     sex         dob intake_date       intake_cond  ...  \
27413             NaN  Spayed  2019-04-16  2021-04-17            NORMAL  ...   
27414           WHITE    Male  2020-05-14  2020-06-14  UNDER AGE/WEIGHT  ...   
27415            GRAY    Male  2021-04-10  2021-11-12            NORMAL  ...   
27416             NaN  Female  2019-08-03  2019-08-24  UNDER AGE/WEIGHT  ...   
27417           WHITE  Female  2017-05-14  2018-02-22            NORMAL  ...   

        intake_is_dead                         geopoint intake

In [32]:
df["zip"] = df["crossing"].str.extract(r"\b(\d{5})(?:-\d{4})?\s*$")[0]

# 2. Raw count (everything in the file)
print("Unique zips, all rows:", df["zip"].nunique())
print("Rows with no zip:", df["zip"].isna().sum(), f"({df['zip'].isna().mean():.1%})")

# 3. Restrict to Long Beach jurisdiction
lb = df[df["jurisdiction"] == "LONG BEACH"]
print("Unique zips, Long Beach jurisdiction:", lb["zip"].nunique())

# 4. See how concentrated the counts are, then drop rare outliers
counts = lb["zip"].value_counts()
print(counts)

MIN_ROWS = 10  # adjust to taste
core = counts[counts >= MIN_ROWS]
print(f"Zips with at least {MIN_ROWS} intakes:", len(core))
print(sorted(core.index))

Unique zips, all rows: 102
Rows with no zip: 2852 (10.4%)
Unique zips, Long Beach jurisdiction: 38
zip
90805    5755
90813    3551
90806    2541
90810    1735
90815    1626
90804    1430
90802    1249
90808    1246
90807    1160
90803     443
90814     414
90703      92
90755      40
90840      25
90801      16
90740       8
90720       7
90822       6
90706       5
90670       5
90712       4
90701       3
90242       3
90262       2
90715       2
90713       2
90640       2
90705       2
90291       1
90277       1
90745       1
90846       1
90623       1
95037       1
90602       1
90731       1
90630       1
90185       1
Name: count, dtype: int64
Zips with at least 10 intakes: 15
['90703', '90755', '90801', '90802', '90803', '90804', '90805', '90806', '90807', '90808', '90810', '90813', '90814', '90815', '90840']
